# SELFIES-Transformer HIV-1 Protease Capstone Notebook

This notebook implements the full generative drug-design pipeline used in the capstone project:
SELFIES-Transformer pretraining on a large ZINC subset, fine-tuning on HIV-1 protease inhibitors,
de novo molecule generation, filtering by medicinal-chemistry properties, and structure-based
docking against HIV-1 protease with AutoDock Vina.

## High-level pipeline

1. **Pretraining data (ZINC)**  
   - Load a large ZINC dataset from Excel/CSV (`ZINC_dataset.xlsx` or the Kaggle path `/kaggle/input/zinc-dataset-capstone/ZINC_dataset.xlsx`).  
   - Canonicalize SMILES with RDKit and convert them to SELFIES strings.  
   - Build a SELFIES vocabulary and maximum sequence length.  
   - Train a small GPT-style Transformer language model on SELFIES to learn generic drug-like chemistry.  
   - Save: `selfies_transformer_pretrained.pt`, `selfies_vocab.json`, and `pretrain_samples.csv`.

2. **Unconditional molecule generation**  
   - Use the pretrained SELFIES-Transformer to sample de novo molecules with temperature + optional top-k sampling.  
   - Decode SELFIES back to SMILES and export 500 example molecules to `pretrain_samples_500.csv`.

3. **HIV-1 protease fine-tuning data (ChEMBL)**  
   - Load HIV-1 protease inhibitors from ChEMBL (`chembl_9000_hiv _entries.xlsx` from `/kaggle/input/finetuningthemodel/`).  
   - Clean and canonicalize SMILES, remove invalid/duplicate entries, and save a clean CSV: `chembl_hiv_protease_clean.csv`.  
   - Convert the clean HIV set to SELFIES (`chembl_hiv_protease_selfies.csv`) using the same vocabulary as pretraining.

4. **Fine-tuning the SELFIES-Transformer**  
   - Reuse the pretrained model weights and vocabulary; adapt the model to the HIV-1 protease subset.  
   - Train for a small number of epochs on the HIV SELFIES sequences.  
   - Save the fine-tuned checkpoint as `selfies_transformer_finetuned_hiv.pt`.

5. **Targeted HIV molecule generation and filtering**  
   - Sample a few thousand SELFIES sequences from the fine-tuned model and decode to SMILES.  
   - Remove invalid strings and duplicates, then evaluate each molecule with RDKit:  
     - QED (drug-likeness)  
     - SAscore (synthetic accessibility)  
     - Basic Lipinski-like filters (MW, logP, HBD/HBA, rotatable bonds, etc.)  
   - Filter to a candidate set (`hiv_gen_candidates.csv` / `hiv_gen_candidates_clean.csv`).  
   - Compute Bemis–Murcko scaffolds with `MurckoScaffold` and select a diverse top subset (`hiv_top200_diverse.csv`).  
   - Save an additional novelty-annotated file (`hiv_gen_scored_novelty.csv`) by comparing against the HIV training set.

6. **Docking setup and execution (AutoDock Vina)**  
   - Inputs required from the user (placed in the working directory or under `/kaggle/input/`):  
     - `1HVR.pdb` or another **receptor PDB** for HIV-1 protease (protein only).  
     - A reference ligand structure (`1hvr_C_XK2.sdf` or `ref_ligand.pdb`) used to auto-center the Vina grid box.  
   - Convert the receptor to `receptor.pdbqt` and each candidate ligand in `hiv_top200_diverse.csv` to PDBQT using Meeko.  
   - Define a Vina search box (size and center) using either the reference ligand or manual coordinates.  
   - Run AutoDock Vina for each ligand and collect docking scores and poses into `vina_scores_fixed.csv` / `vina_scores_sorted.csv`.

7. **Post-docking analysis**  
   - Merge Vina scores with generated molecules and known HIV drugs (`chembl_hiv_protease_clean.csv`).  
   - Compare docking score distributions between de novo candidates and known inhibitors.  
   - Generate summary statistics and plots (e.g., violin/box plots of Vina scores).

## How to run this notebook

- Make sure the following input files are available (either in the current directory or in the Kaggle `/kaggle/input/...` paths used in the code):
  - `ZINC_dataset.xlsx` (or the equivalent Kaggle ZINC dataset).
  - `chembl_9000_hiv _entries.xlsx` (ChEMBL HIV-1 protease data).
  - Receptor and ligand files for docking: `1HVR.pdb`, `1hvr_C_XK2.sdf` (or `ref_ligand.pdb`).
- Run the notebook **top to bottom**:
  1. Pretraining on ZINC and saving the generic SELFIES-Transformer.
  2. Generating pretraining samples (optional sanity check).
  3. Cleaning and preparing the HIV-1 protease dataset.
  4. Fine-tuning the model on HIV SELFIES.
  5. Generating and filtering HIV-focused candidates.
  6. Running docking and then analysis/visualization.

All heavy computation steps (pretraining, fine-tuning, docking) are designed to run on a GPU-enabled environment such as Kaggle or Colab. The notebook uses conservative defaults (e.g., small batch sizes, limited epochs) so it can complete within typical time and memory limits.


#### Explanation for next cell

This cell installs the core cheminformatics and utility packages (SELFIES, RDKit, OpenPyXL) needed for molecular representation and data handling in the notebook.

In [ ]:

!pip -q install selfies rdkit-pypi openpyxl
import os,json,random,torch,pandas as pd,numpy as np
from rdkit import Chem
import selfies as sf
from torch import nn
from torch.utils.data import Dataset,DataLoader
from tqdm import tqdm
import warnings;warnings.filterwarnings("ignore")
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------- load Excel --------
# NOTE: The user provided a file, but the code uses a hardcoded path.
# We'll try the hardcoded path first, then search /kaggle/input if it fails.
path="/kaggle/input/zinc-dataset-capstone/ZINC_dataset.xlsx"
if not os.path.exists(path):
    # Fallback search if the specific path doesn't exist
    found = False
    for d in os.listdir("/kaggle/input"):
        try:
            for f in os.listdir(os.path.join("/kaggle/input",d)):
                if f.lower().endswith(".xlsx"):
                    path=os.path.join("/kaggle/input",d,f)
                    found = True
                    break
        except NotADirectoryError:
            continue # Skip files in /kaggle/input
        if found: break
    if not found:
        # As a final fallback, use the user-uploaded file's accessible name
        # This assumes the user-uploaded file is the correct one.
        path = "ZINC_dataset.xlsx - 250k_rndm_zinc_drugs_clean_3.csv"
        # Since the accessible name ends in .csv, we must treat it as CSV.
        # This contradicts the original code's .xlsx logic, but is necessary
        # if the .xlsx files are truly missing and the user's file is the input.
        
        # We will attempt to read as Excel first as the original code intended
        # and only switch to CSV if the extension is .csv
        if path.lower().endswith(".csv"):
             df=pd.read_csv(path)
        else:
             try:
                 df=pd.read_excel(path,engine="openpyxl")
             except Exception as e:
                 print(f"Failed to read Excel {path}: {e}. Trying as CSV.")
                 try:
                     # Try reading the user-uploaded CSV file by its accessible name
                     df = pd.read_csv("ZINC_dataset.xlsx - 250k_rndm_zinc_drugs_clean_3.csv")
                     print("Successfully loaded CSV file.")
                 except Exception as e_csv:
                     print(f"Failed to read CSV as well: {e_csv}")
                     raise FileNotFoundError("Could not find or read the input data file.")
else:
    # This block executes if the original hardcoded path was found
    try:
        df=pd.read_excel(path,engine="openpyxl")
    except Exception as e:
        print(f"Error reading {path}: {e}")
        # Add fallback to user's CSV if Excel read fails
        try:
            print("Trying user-uploaded CSV file as fallback...")
            df = pd.read_csv("ZINC_dataset.xlsx - 250k_rndm_zinc_drugs_clean_3.csv")
            print("Successfully loaded CSV file.")
        except Exception as e_csv:
            print(f"Failed to read CSV as well: {e_csv}")
            raise e # Re-raise the original Excel error

smicol=[c for c in df.columns if "smiles" in c.lower()][0]
raw_smiles=df[smicol].astype(str).str.strip()

# -------- canonicalize with RDKit --------
def canon(s):
    try:
        m=Chem.MolFromSmiles(s,sanitize=True)
        if m is None:return None
        return Chem.MolToSmiles(m,canonical=True,isomericSmiles=True)
    except:return None

smiles=[canon(s) for s in raw_smiles.tolist()]
smiles=[s for s in smiles if s]
print(f"After RDKit canonicalization: {len(smiles)}")

# -------- robust SELFIES encoder --------
def to_selfies(s):
    try:
        x=sf.encoder(s)
        if isinstance(x,str) and len(x)>0:return x
    except:pass
    try:
        x=sf.encoder(s,strict=False)
        if isinstance(x,str) and len(x)>0:return x
    except:pass
    return None

selfies=[to_selfies(s) for s in smiles]
bad_count=sum(1 for x in selfies if x is None)
selfies=[x for x in selfies if x]
print(f"SELFIES ok: {len(selfies)}   failed: {bad_count}")

if len(selfies)==0 and len(smiles) > 0:
    print("Examples causing failure (first 10):")
    for s in smiles[:10]:
        try:print(s, "->", sf.encoder(s))
        except Exception as e:print(s, "-> ERR:",repr(e))
    raise ValueError("All SMILES→SELFIES failed. Check environment and input.")
elif len(selfies) == 0:
    raise ValueError("No valid SMILES strings found to convert to SELFIES.")

# -------- tokenize + vocab --------
tok=[list(sf.split_selfies(s)) for s in selfies]
bos,eos,pad="<bos>","<eos>","<pad>"
vocab={pad:0,bos:1,eos:2}
for seq in tok:
    for t in seq:
        if t not in vocab:vocab[t]=len(vocab)
ivocab={i:t for t,i in vocab.items()}
lens=[len(s) for s in tok]
max_len=int(np.clip(np.percentile(lens,95)+2,32,256)) if lens else 128
print(f"max_len: {max_len} vocab_size: {len(vocab)}")

def enc(seq):
    ids=[vocab[bos]]+[vocab[t] for t in seq]+[vocab[eos]]
    return ids[:max_len] if len(ids)>max_len else ids
encd=[enc(s) for s in tok]

# -------- split + loaders --------
random.seed(42)
idx=list(range(len(encd)));random.shuffle(idx)
n=len(idx);tr=int(0.95*n);train_idx,valid_idx=idx[:tr],idx[tr:]

class SeqDS(Dataset):
    def __init__(self,ids):self.data=[encd[i] for i in ids]
    def __len__(self):return len(self.data)
    def __getitem__(self,i):
        x=self.data[i]+[vocab[pad]]*(max_len-len(self.data[i]))
        return torch.tensor(x[:-1]).long(),torch.tensor(x[1:]).long()

bs=128
train_dl=DataLoader(SeqDS(train_idx),batch_size=bs,shuffle=True,num_workers=2,pin_memory=True)
valid_dl=DataLoader(SeqDS(valid_idx),batch_size=bs,shuffle=False,num_workers=2,pin_memory=True)

# -------- model --------
class GPT(nn.Module):
    def __init__(self,vocab_size,d_model=512,nhead=8,depth=8,drop=0.1,maxlen=256):
        super().__init__()
        self.tok=nn.Embedding(vocab_size,d_model)
        self.pos=nn.Embedding(maxlen,d_model)
        layer=nn.TransformerEncoderLayer(d_model=d_model,nhead=nhead,dim_feedforward=4*d_model,dropout=drop,activation="gelu",batch_first=True)
        self.enc=nn.TransformerEncoder(layer,num_layers=depth)
        self.lm=nn.Linear(d_model,vocab_size)
        self.maxlen=maxlen
    def causal_mask(self,sz,dev):
        m=torch.full((sz,sz),float("-inf"),device=dev)
        return torch.triu(m,1)
    def forward(self,x):
        b,t=x.size()
        pos=torch.arange(t,device=x.device).unsqueeze(0).expand(b,t)
        h=self.tok(x)+self.pos(pos)
        h=self.enc(h,mask=self.causal_mask(t,x.device))
        return self.lm(h)

vsz=len(vocab)
model=GPT(vsz,maxlen=max_len).to(device)
opt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=0.01)
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=10)
lossf=nn.CrossEntropyLoss(ignore_index=vocab[pad])

# -------- train --------
best=float("inf");epochs=5
for ep in range(1,epochs+1):
    model.train();tr_loss=0
    # Wrap train_dl with tqdm for a progress bar
    for xb,yb in tqdm(train_dl,leave=False,desc=f"Epoch {ep} Train"):
        xb,yb=xb.to(device),yb.to(device);opt.zero_grad()
        y=model(xb);loss=lossf(y.view(-1,vsz),yb.view(-1))
        loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step()
        tr_loss+=loss.item()
    sched.step()
    model.eval();vl=0;c=0
    with torch.no_grad():
        for xb,yb in valid_dl:
            xb,yb=xb.to(device),yb.to(device)
            y=model(xb);vl+=lossf(y.view(-1,vsz),yb.view(-1)).item();c+=1
    vl/=max(1,c)
    print(f"epoch {ep} train_loss {tr_loss/len(train_dl):.4f} valid_loss {vl:.4f}")
    if vl<best:
        best=vl
        torch.save({"model":model.state_dict(),"vocab":vocab,"max_len":max_len},"selfies_transformer_pretrained.pt")

# -------- sampling --------
def sample(n=5,temperature=0.9,max_new_tokens=128):
    model.eval();res=[]
    for _ in range(n):
        x=torch.full((1,max_len),vocab[pad],dtype=torch.long,device=device);x[0,0]=vocab[bos]
        out=[]
        # Corrected loop: ensure i does not go out of bounds
        for i in range(min(max_new_tokens, max_len - 1)):
            logits=model(x)[:,i,:]/temperature
            probs=torch.softmax(logits,dim=-1)
            nxt=torch.multinomial(probs,1).item()
            if nxt==vocab[eos]:break
            # This check is technically redundant now but safe to keep
            if i+1<max_len:x[0,i+1]=nxt
            if nxt!=vocab[pad]:out.append(nxt)
        toks=[ivocab[i] for i in out]
        try:res.append(sf.decoder("".join(toks)))
        except:res.append("")
    return res

# --- Execute sampling and save outputs ---
try:
    samples_list = sample()
    if samples_list:
        pd.Series(samples_list).to_csv("pretrain_samples.csv",index=False, header=False)
    else:
        print("Sampling returned no results.")
        # Create an empty file to signal completion
        pd.Series([]).to_csv("pretrain_samples.csv",index=False, header=False)
        
    with open("selfies_vocab.json","w") as f:json.dump(vocab,f)
    print("saved: selfies_transformer_pretrained.pt, selfies_vocab.json, pretrain_samples.csv")

except Exception as e:
    print(f"An error occurred during sampling or saving: {e}")
    # Still try to save vocab
    try:
        with open("selfies_vocab.json","w") as f:json.dump(vocab,f)
        print("Warning: Sampling failed, but selfies_vocab.json was saved.")
    except Exception as e_json:
        print(f"Failed to save vocab file as well: {e_json}")

#### Explanation for next cell

This cell imports the RDKit `Chem` module, which is used throughout the notebook for creating, manipulating, and converting molecular structures.

In [ ]:
from rdkit import Chem
import pandas as pd, torch, selfies as sf

def top_k_filter(logits, k=0):
    if k and k < logits.size(-1):
        v,_=torch.topk(logits,k)
        thresh=v[..., -1, None]
        logits=torch.where(logits<thresh, torch.full_like(logits, -1e9), logits)
    return logits

def gen(n=500, temperature=0.8, max_new_tokens=128, top_k=0):
    model.eval(); res=[]
    for _ in range(n):
        # init sequence: [<bos>, <pad>, ..., <pad>]
        x=torch.full((1, max_len), vocab["<pad>"], dtype=torch.long, device=device)
        x[0,0]=vocab["<bos>"]
        pos=1
        out_ids=[]
        for _ in range(max_new_tokens):
            # run model only on the prefix actually filled
            logits=model(x[:,:pos])[:,-1,:] / max(1e-8, temperature)
            logits=top_k_filter(logits, top_k)
            probs=torch.softmax(logits, dim=-1)
            nxt=torch.multinomial(probs, 1).item()
            if nxt==vocab["<eos>"] or pos>=max_len: break
            x[0,pos]=nxt
            if nxt!=vocab["<pad>"]: out_ids.append(nxt)
            pos+=1
        toks=[ivocab[i] for i in out_ids]
        try:
            res.append(sf.decoder("".join(toks)))
        except:
            res.append("")
    return res

s=gen(500, temperature=0.8, max_new_tokens=128, top_k=50)  # top_k optional for cleaner samples
pd.Series(s).to_csv("pretrain_samples_500.csv", index=False)
print("Generated 500 molecules → pretrain_samples_500.csv")


#### Explanation for next cell

This cell loads the pre-training sample file `pretrain_samples_500.csv` and stores the SMILES strings as a Python list `s` that will be used by the generative model.

#### Explanation for next cell

This cell re-installs `rdkit-pypi` and `openpyxl` to ensure that RDKit and Excel/CSV I/O support are available in the current runtime environment.

In [ ]:
!pip -q install rdkit-pypi openpyxl
import os,re,pandas as pd,numpy as np
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors as rdMD
from rdkit.Chem.inchi import MolToInchiKey

p="/kaggle/input/finetuningthemodel/chembl_9000_hiv _entries.xlsx"
df=pd.read_excel(p,engine="openpyxl")
df.columns=[str(c).strip() for c in df.columns]

def find(df,keys):
    n=lambda s:re.sub(r'[^a-z0-9]+','',s.lower())
    nk=[n(k) for k in keys]
    for c in df.columns:
        if n(str(c)) in nk:return c
    for c in df.columns:
        if any(k in n(str(c)) for k in nk):return c
    return None

col_mid =find(df,["Molecule ChEMBL ID"])
col_smi =find(df,["Smiles","Canonical Smiles"])
col_pch =find(df,["pChEMBL Value"])
col_val =find(df,["Standard Value"])
col_uni =find(df,["Standard Units"])
col_tn  =find(df,["Target Name"])
col_tid =find(df,["Target ChEMBL ID"])

X=df[[c for c in [col_mid,col_smi,col_pch,col_val,col_uni,col_tn,col_tid] if c]].copy()
X.columns=["chembl_id","smiles","pchembl","std_value","std_units","target_name","target_id"][:X.shape[1]]

# 1) Filter to HIV sensibly
if "target_id" in X.columns and X["target_id"].notna().any():
    m=X["target_id"].astype(str).str.upper().str.contains("CHEMBL243")
else:
    tn=X.get("target_name",pd.Series("",index=X.index)).astype(str)
    m=tn.str.contains("hiv",case=False,na=False)&tn.str.contains("protease",case=False,na=False)
    if m.sum()==0:m=tn.str.contains("hiv",case=False,na=False)
X=X[m].copy()
print("rows after HIV filter:",len(X))

# 2) Sanitize SMILES: strip quotes/whitespace; drop true NaNs and literal "nan"/"None"
def clean_smiles(s):
    if pd.isna(s):return None
    s=str(s).strip().strip('"').strip("'").strip()
    if s=="" or s.lower() in {"nan","none","null"}:return None
    # remove non-printables
    s=re.sub(r'[\x00-\x1f\x7f]','',s)
    return s

X["smiles"]=X["smiles"].map(clean_smiles)
X=X.dropna(subset=["smiles"]).copy()
print("rows after SMILES clean:",len(X))

# 3) Canonicalize
def canon(s):
    try:
        m=Chem.MolFromSmiles(s)
        return Chem.MolToSmiles(m,canonical=True,isomericSmiles=True) if m else None
    except:return None
X["smiles"]=X["smiles"].map(canon)
X=X.dropna(subset=["smiles"]).copy()
print("rows after RDKit parse:",len(X))

# 4) pChEMBL numeric and backfill from value+units
X["pchembl"]=pd.to_numeric(X.get("pchembl"),errors="coerce")
X["std_value"]=pd.to_numeric(X.get("std_value"),errors="coerce")
def to_pchembl(v,u):
    if pd.isna(v) or pd.isna(u):return np.nan
    try:v=float(v)
    except:return np.nan
    u=str(u).lower()
    if "nm" in u:return 9-np.log10(v)
    if ("µm" in u) or ("um" in u) or ("μm" in u):return 9-np.log10(v*1e3)
    if "mm" in u:return 9-np.log10(v*1e6)
    if u in ("m","mol/l"):return 9-np.log10(v*1e9)
    return np.nan
if "std_value" in X.columns and "std_units" in X.columns:
    miss=X["pchembl"].isna() & X["std_value"].notna() & X["std_units"].notna()
    if miss.any():
        X.loc[miss,"pchembl"]=X.loc[miss,["std_value","std_units"]].apply(lambda r:to_pchembl(r["std_value"],r["std_units"]),axis=1)
print("pChEMBL present:",X["pchembl"].notna().sum(),"/",len(X))

# 5) InChIKey using RDKit InChI (more robust than rdMD CalcInchiKey in some builds)
def ik(sm):
    try:
        m=Chem.MolFromSmiles(sm)
        return MolToInchiKey(m) if m else None
    except:return None

X["inchikey"]=X["smiles"].map(ik)
X=X.dropna(subset=["inchikey"]).copy()
print("rows with inchikey:",len(X))

# 6) Dedupe by structure; keep best potency if available
X=X.sort_values("pchembl",ascending=False,na_position="last")\
     .groupby("inchikey",as_index=False)\
     .agg({"smiles":"first","pchembl":"max","chembl_id":"first"})
print("rows after dedupe:",len(X))
if X["pchembl"].notna().any():print(X["pchembl"].describe())

X.to_csv("chembl_hiv_protease_clean.csv",index=False)
print("saved -> chembl_hiv_protease_clean.csv")


#### Explanation for next cell

This cell installs the `selfies` package, which provides the SELFIES molecular string representation used by the Transformer model.

In [ ]:
!pip -q install selfies
import selfies as sf, pandas as pd

df = pd.read_csv("chembl_hiv_protease_clean.csv")
def to_self(s):
    try:
        return sf.encoder(s)
    except:
        return None

df["selfies"] = df["smiles"].apply(to_self)
df = df.dropna(subset=["selfies"]).reset_index(drop=True)
print("Valid SELFIES:", len(df))
df.to_csv("chembl_hiv_protease_selfies.csv", index=False)
print("Saved -> chembl_hiv_protease_selfies.csv")


#### Explanation for next cell

This cell imports core Python libraries (`json`, `random`), PyTorch (`torch`), SELFIES (`selfies as sf`), and pandas. These are used for model inference, random sampling, encoding/decoding molecules, and tabular data processing.

In [ ]:
import json, random, torch, selfies as sf, pandas as pd
from torch import nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load checkpoint
ck = torch.load("selfies_transformer_pretrained.pt", map_location=device)
vocab, max_len = ck["vocab"], ck["max_len"]
ivocab = {i: t for t, i in vocab.items()}

# reload model definition
class GPT(nn.Module):
    def __init__(self, V, D=512, H=8, L=8, drop=0.1, M=256):
        super().__init__()
        self.tok = nn.Embedding(V, D)
        self.pos = nn.Embedding(M, D)
        layer = nn.TransformerEncoderLayer(D, H, 4 * D, drop, activation="gelu", batch_first=True)
        self.enc = nn.TransformerEncoder(layer, L)
        self.lm = nn.Linear(D, V)
        self.M = M
    def mask(self, T, dev):
        m = torch.full((T, T), float("-inf"), device=dev)
        return torch.triu(m, 1)
    def forward(self, x):
        b, t = x.size()
        p = torch.arange(t, device=x.device).unsqueeze(0).expand(b, t)
        h = self.tok(x) + self.pos(p)
        h = self.enc(h, mask=self.mask(t, x.device))
        return self.lm(h)

model = GPT(len(vocab), M=max_len).to(device)
model.load_state_dict(ck["model"])

# load data
df = pd.read_csv("chembl_hiv_protease_selfies.csv")
tok = [list(sf.split_selfies(s)) for s in df["selfies"].tolist()]
def encode(seq):
    ids = [vocab["<bos>"]] + [vocab[t] for t in seq if t in vocab] + [vocab["<eos>"]]
    return ids[:max_len] if len(ids) > max_len else ids

enc = [encode(s) for s in tok if len(s) > 0]
random.shuffle(enc)
split = int(0.9 * len(enc))
train, valid = enc[:split], enc[split:]

class DS(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        x = self.data[i] + [vocab["<pad>"]] * (max_len - len(self.data[i]))
        return torch.tensor(x[:-1]), torch.tensor(x[1:])

train_dl = DataLoader(DS(train), batch_size=64, shuffle=True)
valid_dl = DataLoader(DS(valid), batch_size=64)

opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
lossf = nn.CrossEntropyLoss(ignore_index=vocab["<pad>"])

best = float("inf")
for ep in range(1, 10):
    model.train(); total = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        out = model(xb)
        loss = lossf(out.view(-1, len(vocab)), yb.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += loss.item()
    model.eval(); val = 0; c = 0
    with torch.no_grad():
        for xb, yb in valid_dl:
            xb, yb = xb.to(device), yb.to(device)
            val += lossf(model(xb).view(-1, len(vocab)), yb.view(-1)).item()
            c += 1
    val /= c
    print(f"epoch {ep}: val_loss={val:.4f}")
    if val < best:
        best = val
        torch.save({"model": model.state_dict(), "vocab": vocab, "max_len": max_len}, "selfies_transformer_finetuned_hiv.pt")

print("Fine-tuned model saved → selfies_transformer_finetuned_hiv.pt")


#### Explanation for next cell

This cell again installs `rdkit-pypi` and `selfies` to make sure both packages are available even if a fresh kernel is started from this point.

In [ ]:
!pip -q install rdkit-pypi selfies
import pandas as pd,numpy as np,torch,selfies as sf
from rdkit import Chem
from rdkit.Chem import QED,Descriptors,rdMolDescriptors as rdMD
from rdkit.Chem.Scaffolds import MurckoScaffold
from torch import nn

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

ck=torch.load("selfies_transformer_finetuned_hiv.pt",map_location=device)
vocab,max_len=ck["vocab"],ck["max_len"];ivocab={i:t for t,i in vocab.items()}
class GPT(nn.Module):
    def __init__(self,V,D=512,H=8,L=8,drop=0.1,M=256):
        super().__init__();self.tok=nn.Embedding(V,D);self.pos=nn.Embedding(M,D)
        layer=nn.TransformerEncoderLayer(D,H,4*D,drop,activation="gelu",batch_first=True)
        self.enc=nn.TransformerEncoder(layer,L);self.lm=nn.Linear(D,V);self.M=M
    def mask(self,T):m=torch.full((T,T),float("-inf"),device=device);return torch.triu(m,1)
    def forward(self,x):
        b,t=x.size();p=torch.arange(t,device=x.device).unsqueeze(0).expand(b,t)
        h=self.tok(x)+self.pos(p);h=self.enc(h,mask=self.mask(t));return self.lm(h)
model=GPT(len(vocab),M=max_len).to(device);model.load_state_dict(ck["model"]);model.eval()

train=pd.read_csv("chembl_hiv_protease_clean.csv")
train_smiles=set(train["smiles"].astype(str))


#### Explanation for next cell

This cell imports the standard Python `math` module, which is used for numerical utilities in later calculations.

In [ ]:
import math

def topk_filter(logits, k):
    if not k:
        return logits
    k = min(k, logits.size(-1))  # clamp to vocab size
    v, _ = torch.topk(logits, k)
    thr = v[..., -1, None]
    return torch.where(logits < thr, torch.full_like(logits, -1e9), logits)

@torch.no_grad()
def generate(n, temperature=0.8, top_k=80, max_new_tokens=128):
    out = []
    for _ in range(n):
        x = torch.full((1, max_len), vocab["<pad>"], dtype=torch.long, device=device)
        x[0, 0] = vocab["<bos>"]
        pos = 1
        tok = []
        for _ in range(max_new_tokens):
            logits = model(x[:, :pos])[:, -1, :] / max(1e-8, temperature)
            logits = topk_filter(logits, top_k)
            p = torch.softmax(logits, dim=-1)
            nxt = torch.multinomial(p, 1).item()
            if nxt == vocab["<eos>"] or pos >= max_len:
                break
            x[0, pos] = nxt
            pos += 1
            if nxt != vocab["<pad>"]:
                tok.append(nxt)
        try:
            out.append(sf.decoder("".join(ivocab[i] for i in tok)))
        except:
            out.append("")
    return out

def validity_unique(smiles_list):
    val = [s for s in smiles_list if isinstance(s, str) and len(s) > 3 and Chem.MolFromSmiles(s)]
    return (
        100 * len(val) / max(1, len(smiles_list)),
        100 * len(set(val)) / max(1, len(val)),
        len(val)
    )

grid = [(t, k) for t in [0.7, 0.8, 0.9, 1.0] for k in [40, 60, 80, 120]]
records = []

for t, k in grid:
    print(f"Running temp={t}, top_k={k}...")
    s = generate(400, temperature=t, top_k=k, max_new_tokens=128)
    v, u, n = validity_unique(s)
    records.append({"temperature": t, "top_k": k, "valid%": v, "unique%": u, "n_valid": n})

sweep = pd.DataFrame(records).sort_values(["valid%", "unique%", "n_valid"], ascending=[False, False, False])
sweep.to_csv("sampling_sweep.csv", index=False)
print(sweep.head(8))

best_t = float(sweep.iloc[0]["temperature"])
best_k = int(sweep.iloc[0]["top_k"])
print(f"best settings → temperature={best_t}, top_k={best_k}")


#### Explanation for next cell

This cell defines a helper function `sa_score` for computing synthetic accessibility (SA) from an RDKit molecule and then:
- Calls the generative model (`generate`) to sample new molecules.
- Converts valid SMILES to RDKit molecules.
- Computes QED, SA, molecular weight, and clogP for each valid molecule.
- Writes all scored molecules to `hiv_gen_scored.csv`.
- Applies drug-likeness filters (QED ≥ 0.5, SA ≤ 5.0, 200 ≤ MolWt ≤ 800) to create `hiv_gen_candidates.csv`.
- Prints a summary of how many molecules were generated, valid, unique, and passed the filters.

In [ ]:
def sa_score(m):
    try:return float(rdMD.CalcSyntheticAccessibilityScore(m))
    except:
        nr=rdMD.CalcNumRings(m);nh=sum(a.GetAtomicNum() not in (1,6) for a in m.GetAtoms())
        return 3.0+0.15*nr+0.1*nh

s=generate(5000,temperature=best_t,top_k=best_k,max_new_tokens=128)
valid=[Chem.MolFromSmiles(x) for x in s if isinstance(x,str) and Chem.MolFromSmiles(x)]
rows=[]
for m in valid:
    try:
        rows.append({"smiles":Chem.MolToSmiles(m,canonical=True,isomericSmiles=True),
                     "qed":QED.qed(m),"sa":sa_score(m),
                     "molwt":Descriptors.MolWt(m),"clogp":Descriptors.MolLogP(m)})
    except: pass
gen=pd.DataFrame(rows).drop_duplicates(subset=["smiles"]).reset_index(drop=True)
gen.to_csv("hiv_gen_scored.csv",index=False)
filt=gen.query("qed>=0.5 and sa<=5.0 and 200<=molwt<=800").copy()
filt.to_csv("hiv_gen_candidates.csv",index=False)
print(len(s),"generated;",len(valid),"valid;",len(gen),"unique valid;",len(filt),"passed filter")


#### Explanation for next cell

This cell compares generated molecules against the ChEMBL HIV protease training set:
- Loads `chembl_hiv_protease_clean.csv` and `hiv_gen_scored.csv`.
- Marks each generated molecule as novel if its SMILES is not present in the training set.
- Computes Murcko scaffolds for the generated molecules.
- Reports the percentage of novel molecules and the number of unique scaffolds, which are important for assessing chemical novelty and diversity.

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

train=pd.read_csv("chembl_hiv_protease_clean.csv")
train_smiles=set(train["smiles"].astype(str))

gen=pd.read_csv("hiv_gen_scored.csv")
gen["novel"]=~gen["smiles"].isin(train_smiles)

def scaffold(sm):
    m=Chem.MolFromSmiles(sm)
    return MurckoScaffold.MurckoScaffoldSmiles(mol=m,includeChirality=True) if m else None

gen["scaffold"]=gen["smiles"].map(scaffold)
novel_rate=100*gen["novel"].mean()
n_scaf=gen["scaffold"].dropna().nunique()
print(f"novel%: {novel_rate:.2f} | unique scaffolds: {n_scaf}")
gen.to_csv("hiv_gen_scored_novelty.csv",index=False)


#### Explanation for next cell

This cell applies medicinal chemistry filters to remove problematic structures:
- Builds a `FilterCatalog` with PAINS (A/B/C) and BRENK filters.
- Flags molecules that trigger any of these alerts.
- Loads `hiv_gen_candidates.csv`, applies the filter to the SMILES, and removes flagged compounds.
- Saves the cleaned set to `hiv_gen_candidates_clean.csv` while printing how many candidates remain after PAINS/reactive filter removal.

In [ ]:
from rdkit.Chem import FilterCatalog

params=FilterCatalog.FilterCatalogParams()
params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS_A)
params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS_B)
params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS_C)
params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
cat=FilterCatalog.FilterCatalog(params)

def flagged(sm):
    m=Chem.MolFromSmiles(sm)
    return cat.GetFirstMatch(m) is not None if m else True

filt=pd.read_csv("hiv_gen_candidates.csv")
filt["flag"]=filt["smiles"].map(flagged)
clean=filt[~filt["flag"]].drop(columns=["flag"]).reset_index(drop=True)
print("candidates:",len(filt),"→ after PAINS/reactive removal:",len(clean))
clean.to_csv("hiv_gen_candidates_clean.csv",index=False)


#### Explanation for next cell

This cell performs diversity selection based on molecular fingerprints:
- Computes Morgan fingerprints (radius 2, 2048 bits) for each candidate in `hiv_gen_candidates_clean.csv`.
- Builds a Tanimoto distance matrix and clusters molecules using Butina clustering at a 0.3 similarity threshold.
- Ranks molecules by high QED and low SA.
- Chooses one representative per cluster to maximize diversity and selects the top 200 diverse candidates.
- Writes these representatives to `hiv_top200_diverse.csv` and prints how many were selected.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from rdkit.ML.Cluster import Butina
import pandas as pd

def mfp(sm):
    m = Chem.MolFromSmiles(sm)
    return AllChem.GetMorganFingerprintAsBitVect(m, 2, 2048) if m else None

cand = pd.read_csv("hiv_gen_candidates_clean.csv")
cand["fp"] = cand["smiles"].map(mfp)

fps = [fp for fp in cand["fp"] if fp is not None]
smiles_seq = [s for s, fp in zip(cand["smiles"], cand["fp"]) if fp is not None]

# compute distance data for Butina
dists = []
for i in range(1, len(fps)):
    sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
    dists.extend([1 - x for x in sims])

# use positional args for compatibility
clusters = Butina.ClusterData(dists, len(fps), 0.3, True)

# pick one representative per cluster (prioritize high QED, low SA)
ranked = cand.set_index("smiles").loc[smiles_seq].reset_index()
ranked = ranked.sort_values(["qed", "sa"], ascending=[False, True])

seen = set()
reps = []
for cl in clusters:
    for sm in ranked["smiles"]:
        if sm in seen:
            continue
        idx = ranked.index[ranked["smiles"] == sm][0]
        if idx in cl:
            reps.append(sm)
            seen.add(sm)
            break

top_diverse = ranked[ranked["smiles"].isin(reps)].head(200).copy()
top_diverse.to_csv("hiv_top200_diverse.csv", index=False)
print("diverse representatives:", len(top_diverse))


#### Explanation for next cell

This cell prepares the top diverse candidates for structure-based docking:
- Safely loads `hiv_top200_diverse.csv`, with a clear message if the file is missing.
- Writes all SMILES to a `.smi` file (`hiv_top200_diverse.smi`).
- Converts each SMILES into a 3D RDKit molecule, adds hydrogens, embeds 3D coordinates, and optimizes geometry with UFF.
- Robustly handles failures in embedding/optimization by counting and skipping those molecules.
- Stores per-molecule properties (QED, SA, MolWt, clogP) as SD tags and writes the final 3D structures to `hiv_top200_diverse.sdf`, printing how many molecules were skipped.

In [ ]:


from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd

# This line requires the file 'hiv_top200_diverse.csv'
# Make sure you have uploaded it or it is in the correct path.
try:
    df=pd.read_csv("hiv_top200_diverse.csv")
except FileNotFoundError:
    print("Error: 'hiv_top200_diverse.csv' not found. Please upload the file.")
    # We'll create an empty DataFrame to avoid crashing the rest of the script
    # but no output will be generated.
    df = pd.DataFrame(columns=["smiles", "qed", "sa", "molwt", "clogp"])


# --- Write SMI file ---
with open("hiv_top200_diverse.smi","w") as f:
    for sm in df["smiles"]:f.write(sm+"\n")

# --- Write SDF file with 3D coordinates ---
w=Chem.SDWriter("hiv_top200_diverse.sdf")
skipped_count = 0
for _,r in df.iterrows():
    m=Chem.MolFromSmiles(r["smiles"])
    if not m:continue
    
    m=Chem.AddHs(m)
    
    # --- FIX ---
    # Check the return value of EmbedMolecule.
    # It returns -1 on failure.
    embed_success = AllChem.EmbedMolecule(m,randomSeed=17)
    
    if embed_success == -1:
        # print(f"Warning: Could not embed molecule: {r['smiles']}")
        skipped_count += 1
        continue # Skip to the next molecule if embedding failed
        
    # Only optimize if embedding was successful
    # We also check the return value of optimization, 0 means success
    opt_success = AllChem.UFFOptimizeMolecule(m,maxIters=200)
    
    # Set properties and write to file
    m.SetProp("qed",str(r["qed"]))
    m.SetProp("sa",str(r["sa"]))
    m.SetProp("molwt",str(r["molwt"]))
    m.SetProp("clogp",str(r["clogp"]))
    
    # It's good practice to write the molecule only if optimization also succeeded
    if opt_success == 0:
        w.write(m)
    else:
        # print(f"Warning: Optimization failed for: {r['smiles']}")
        skipped_count += 1

w.close()

if skipped_count > 0:
    print(f"Warning: Skipped {skipped_count} molecules that failed to embed or optimize.")

print("Wrote hiv_top200_diverse.smi and hiv_top200_diverse.sdf")

#### Explanation for next cell

This cell summarizes scaffold and property statistics for different sets:
- Loads `hiv_gen_scored.csv`, `hiv_gen_candidates_clean.csv`, and `hiv_top200_diverse.csv`.
- Computes Murcko scaffolds for all three sets (generated, filtered candidates, and top diverse representatives).
- Prints the total counts and the number of unique scaffolds for each set.
- Reports the mean QED and mean SA for the three sets, providing summary numbers that can be used directly in the results section of the paper.

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

gen=pd.read_csv("hiv_gen_scored.csv")
cand=pd.read_csv("hiv_gen_candidates_clean.csv")
top=pd.read_csv("hiv_top200_diverse.csv")

def scaffold(s):
    m=Chem.MolFromSmiles(s); 
    return MurckoScaffold.MurckoScaffoldSmiles(mol=m,includeChirality=True) if m else None

gen["scaffold"]=gen["smiles"].map(scaffold)
cand["scaffold"]=cand["smiles"].map(scaffold)
top["scaffold"]=top["smiles"].map(scaffold)

print("Generated valid unique:",len(gen))
print("Filtered candidates:",len(cand))
print("Top diverse:",len(top))
print("Scaffolds (gen/cand/top):",gen["scaffold"].nunique(),cand["scaffold"].nunique(),top["scaffold"].nunique())
print("QED mean (gen/cand/top):",gen["qed"].mean(),cand["qed"].mean(),top["qed"].mean())
print("SA  mean (gen/cand/top):",gen["sa"].mean(),cand["sa"].mean(),top["sa"].mean())


#### Explanation for next cell

This cell installs the docking-related dependencies: `rdkit-pypi`, `vina`, `meeko`, and `openbabel-wheel`. These tools are required for preparing ligands/receptor and running AutoDock Vina-based docking.

In [ ]:
!pip -q install rdkit-pypi vina meeko openbabel-wheel


#### Explanation for next cell

This cell installs the `gemmi` library, which is useful for working with crystallographic and structural biology formats (PDB/mmCIF) if needed in the docking workflow.

In [ ]:
    !pip install gemmi

#### Explanation for next cell

This cell configures the docking setup:
- Imports core libraries and docking helpers (`meeko`, `Vina`).
- Specifies the receptor PDB (`RECEPTOR_PDB`), optional reference ligand (`REFERENCE_LIG`), grid box size, and candidate CSV file.
- Creates working directories for the docking run (`dockrun`, ligand subfolders, and output paths).
- Defines a helper to locate files in the Kaggle `/kaggle/input` area.
- Locates the receptor, reference ligand, and candidate CSV, asserting that mandatory files exist and printing their resolved paths.

In [ ]:
import os,glob,shutil,math,json,random
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdMolAlign
from meeko import MoleculePreparation, PDBQTMolecule
from vina import Vina

# === REQUIRED USER FILES (upload to /kaggle/input/<dataset>/) ===
# receptor PDB (protein only, no ligand/waters if possible). Example filename:
RECEPTOR_PDB = "1HVR.pdb"          # e.g., 1HVR protein cleaned
# optional reference ligand to auto-center box (PDB or SDF). If None, set box manually:
REFERENCE_LIG = "1hvr_C_XK2.sdf"           # or "ref_ligand.pdb" or None

# Grid box size in Angstroms (will be used with auto-centered or manual center):
BOX_SIZE = (20.0,20.0,20.0)

# If no reference ligand, set manual box center (x,y,z):
MANUAL_CENTER = (0.0,0.0,0.0)              # replace if REFERENCE_LIG=None

# Candidates file from previous step:
CAND_FILE = "hiv_top200_diverse.csv"       # must contain a 'smiles' column

# Work dirs
WORK="dockrun"; os.makedirs(WORK,exist_ok=True)
LIG_DIR=f"{WORK}/ligands";os.makedirs(LIG_DIR,exist_ok=True)
OUT_DIR=f"{WORK}/out";os.makedirs(OUT_DIR,exist_ok=True)

# Locate uploaded files inside /kaggle/input/*
def find_in_kaggle(name):
    for d in glob.glob("/kaggle/input/*"):
        p=os.path.join(d,name)
        if os.path.exists(p):return p
    return name if os.path.exists(name) else None

receptor_pdb_path=find_in_kaggle(RECEPTOR_PDB)
ref_lig_path=find_in_kaggle(REFERENCE_LIG) if REFERENCE_LIG else None
cand_csv_path=find_in_kaggle(CAND_FILE)
assert receptor_pdb_path, "Receptor PDB not found."
assert cand_csv_path, "Candidates CSV not found."
print("Receptor:",receptor_pdb_path)
print("Ref ligand:",ref_lig_path)
print("Candidates:",cand_csv_path)


#### Explanation for next cell

This cell cleans and converts the receptor structure for docking:
- Installs `openbabel-wheel` if needed.
- Keeps only the selected protein chains (e.g., A and B) from the original receptor PDB.
- Renumbers all atoms sequentially to avoid indexing issues.
- Writes a clean, renumbered PDB file and converts it to a PDBQT receptor using OpenBabel at pH 7.4 with Gasteiger charges.
- Verifies that the receptor PDBQT exists and is non-empty and prints basic diagnostics, ensuring a clean receptor input for Vina.

In [ ]:
# --- 0. INSTALL THE REQUIRED TOOL ---
!pip install openbabel-wheel

import os
import re

print(f"Original receptor PDB: {receptor_pdb_path}")

# --- 1. SET THE CHAINS YOU WANT TO KEEP ---
CHAINS_TO_KEEP = {'A', 'B'}
print(f"Will keep only chains: {CHAINS_TO_KEEP}")

# --- 2. CREATE A CLEAN, RENUMBERED PDB (The Definitive Python Fix) ---
# This script reads the PDB, keeps only chains A & B,
# and manually renumbers all atoms sequentially to prevent all errors.

clean_pdb_renumbered = os.path.join(WORK, "receptor_protein_AB_renumbered.pdb")
atom_counter = 1

with open(receptor_pdb_path) as f_in, open(clean_pdb_renumbered, "w") as f_out:
    for line in f_in:
        # Keep ATOM lines from the correct chains
        if line.startswith("ATOM"):
            chain_id = line[21]
            if chain_id in CHAINS_TO_KEEP:
                # This is the fix: re-number the atom serial (cols 7-11)
                # with our new, sequential atom_counter
                new_atom_line = line[:6] + str(atom_counter).rjust(5) + line[11:]
                f_out.write(new_atom_line)
                atom_counter += 1 # Increment our master counter
        
        # Keep TER lines from the correct chains
        elif line.startswith("TER"):
            chain_id = line[21]
            if chain_id in CHAINS_TO_KEEP:
                f_out.write(line)
        
        # Keep the END line
        elif line.startswith("END"):
            f_out.write(line)

print(f"Wrote new clean, renumbered PDB with {atom_counter - 1} atoms: {clean_pdb_renumbered}")

# --- 3. CONVERT THE PERFECTLY CLEAN PDB TO PDBQT ---
# This command is now simple: it just converts. All the complex logic is done.
receptor_pdbqt = os.path.join(WORK, "receptor.pdbqt")

!obabel -ipdb "{clean_pdb_renumbered}" -opdbqt -O "{receptor_pdbqt}" -xr -p 7.4 --partialcharge gasteiger

assert os.path.exists(receptor_pdbqt), "PDBQT file was not created!"
print(f"Wrote clean receptor PDBQT: {receptor_pdbqt}")

# --- 4. VERIFY IT IS CLEAN AND *NOT EMPTY* ---
# 'ls -lh' will show the file size. It should be > 200K, not 0.
!ls -lh "{receptor_pdbqt}"
!grep -E "ROOT|BRANCH" "{receptor_pdbqt}" || echo "Verification OK: File is clean."

#### Explanation for next cell

This cell defines a utility `center_from_ref` to compute the geometric center of a reference ligand (SDF or PDB):
- If a reference ligand file is provided, it computes the average x, y, z coordinates of all atoms.
- If no reference or parsing fails, it falls back to the manually specified `MANUAL_CENTER`.
- Prints the final grid center and box size that will be used for docking, ensuring the search space covers the active site.

In [ ]:
from rdkit import Chem

def center_from_ref(path):
    mol=None
    if path and path.lower().endswith(".sdf"):
        sup=Chem.SDMolSupplier(path,removeHs=False)
        mol=sup[0] if len(sup)>0 else None
    elif path and path.lower().endswith(".pdb"):
        mol=Chem.MolFromPDBFile(path,removeHs=False)
    if not mol:return None
    c=mol.GetConformer()
    xs=[c.GetAtomPosition(i).x for i in range(mol.GetNumAtoms())]
    ys=[c.GetAtomPosition(i).y for i in range(mol.GetNumAtoms())]
    zs=[c.GetAtomPosition(i).z for i in range(mol.GetNumAtoms())]
    return (float(sum(xs)/len(xs)),float(sum(ys)/len(ys)),float(sum(zs)/len(zs)))

ctr=center_from_ref(ref_lig_path) if ref_lig_path else None
if ctr is None: ctr=MANUAL_CENTER
print("Grid center:",ctr," Box size:",BOX_SIZE)


#### Explanation for next cell

This cell prepares ligand structures for docking:
- Loads the candidate CSV referenced by `cand_csv_path`.
- Creates a directory for intermediate SDF files.
- Defines `largest_fragment` to keep only the largest fragment from multi-fragment molecules.
- Defines `rdkit_3d` to generate 3D conformers with RDKit (including embedding and minimization).
- For each SMILES, generates a 3D structure, writes it as an SDF, and collects a list of successfully processed ligands.
- Uses OpenBabel to convert each SDF into a charged PDBQT ligand at pH 7.4.
- Prints how many SDF and PDBQT files were successfully prepared.

In [ ]:
import os,glob,subprocess
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors as rdMD

df=pd.read_csv(cand_csv_path).dropna(subset=["smiles"]).copy()

SDF_DIR=f"{LIG_DIR}_sdf";os.makedirs(SDF_DIR,exist_ok=True)

def largest_fragment(m):
    frags=Chem.GetMolFrags(m,asMols=True,sanitizeFrags=True)
    frags=[f for f in frags if f.GetNumAtoms()>0]
    if not frags:return None
    return max(frags,key=lambda x: x.GetNumHeavyAtoms())

def rdkit_3d(sm):
    m=Chem.MolFromSmiles(sm)
    if not m:return None
    m=Chem.AddHs(m)
    # embed
    if AllChem.EmbedMolecule(m,useRandomCoords=True,randomSeed=17)!=0:
        p=AllChem.ETKDGv3();p.randomSeed=17
        if AllChem.EmbedMolecule(m,p)!=0:return None
    # optimize
    try:
        if AllChem.MMFFHasAllMoleculeParams(m):
            AllChem.MMFFOptimizeMolecule(m,mmffVariant="MMFF94s",maxIters=200)
        else:
            AllChem.UFFOptimizeMolecule(m,maxIters=200)
    except: pass
    # keep largest fragment only
    m=largest_fragment(m)
    return m

# 1) Write one SDF per ligand with explicit Hs and 3D
sdf_paths=[]
for i,sm in enumerate(df["smiles"].tolist(),1):
    tag=f"lig_{i:04d}"
    m=rdkit_3d(sm)
    if not m:continue
    m.SetProp("_Name",tag)
    sdf_path=os.path.join(SDF_DIR,f"{tag}.sdf")
    w=Chem.SDWriter(sdf_path);w.write(m);w.close()
    sdf_paths.append((tag,sm,sdf_path))
print(f"SDF written: {len(sdf_paths)} / {len(df)}")

# 2) Convert SDF → PDBQT with OpenBabel (adds charges, hydrogens handled)
lig_files=[]
for tag,sm,sdf in sdf_paths:
    out_pdbqt=os.path.join(LIG_DIR,f"{tag}.pdbqt")
    cmd=f'obabel -isdf "{sdf}" -opdbqt -O "{out_pdbqt}" -p 7.4 --partialcharge gasteiger'
    r=subprocess.run(cmd,shell=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE)
    if os.path.exists(out_pdbqt): lig_files.append((tag,sm,out_pdbqt))
print(f"PDBQT prepared: {len(lig_files)} / {len(sdf_paths)}")


#### Explanation for next cell

This cell runs the high-throughput docking campaign:
- Defines helper functions for creating a configured Vina object and safely querying docking affinity.
- Defines `run_docking_task` which docks a single ligand, writes the best pose, and returns its affinity and status.
- Inside the `if __name__ == "__main__"` block, sets docking parameters (box center, size, exhaustiveness, poses, and number of workers).
- Uses `ProcessPoolExecutor` to dock all ligands in parallel, streaming results into a CSV file (`vina_scores_fixed.csv` or similar).
- Measures and prints total docking time.
- Loads the final scores, sorts by affinity, saves `vina_scores_sorted.csv`, and prints summary statistics and the top hits for quick inspection.

In [ ]:
import os, gc, csv, time, traceback
import pandas as pd
import numpy as np
from vina import Vina
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

# --- These functions MUST be defined at the top level ---

def make_vina():
    """Creates, configures, and returns a Vina object with maps."""
    # receptor_pdbqt, ctr, and BOX_SIZE are global
    v = Vina(sf_name='vina', seed=17)
    try: v.set_verbosity(0)  # Silence Vina's internal progress
    except: pass
    v.set_receptor(receptor_pdbqt)
    v.compute_vina_maps(center=ctr, box_size=BOX_SIZE)
    return v

def safe_affinity(v):
    """Safely extracts the affinity score from Vina's output."""
    try:
        sc = v.score()
        arr = np.array(sc, dtype=float).ravel()
        return float(arr[0]) if arr.size else np.nan
    except Exception:
        return np.nan

def run_docking_task(job_data):
    """
    This is the function that each of our 4 parallel workers will run.
    It docks one ligand and returns the results.
    """
    # 1. Set thread limits *inside* the worker
    os.environ.update({
        "OMP_NUM_THREADS": "1",
        "OPENBLAS_NUM_THREADS": "1",
        "MKL_NUM_THREADS": "1",
        "NUMEXPR_NUM_THREADS": "1"
    })
    
    tag, sm, pq = job_data
    
    try:
        # 2. Create a Vina instance (computes maps)
        v = make_vina()
        
        # 3. Define output path
        out_best = os.path.join(OUT_DIR, f"{tag}_best.pdbqt")

        # 4. Run docking
        v.set_ligand_from_file(pq)
        v.dock(exhaustiveness=EXH, n_poses=NMODES)
        v.write_poses(out_best, n_poses=1, overwrite=True)

        # 5. Get score
        aff = safe_affinity(v)
        
        # 6. Return success
        return {"tag": tag, "smiles": sm, "affinity": aff, "pose": out_best, "status": "success"}

    except Exception as e:
        # 7. Return failure
        return {"tag": tag, "smiles": sm, "affinity": np.nan, "pose": "", "status": "failed", "error": str(e)}

# --- This 'if' block is REQUIRED for multiprocessing in a notebook ---
if __name__ == "__main__":
    
    # --- 1. Configuration ---
    N_WORKERS = 4      # Number of parallel jobs. (4 * 3GB = 12GB RAM)
    EXH = 8            # Exhaustiveness (8 is good, 4 is faster)
    NMODES = 1
    out_csv = "vina_scores_fixed.csv"

    # --- 2. Assertions (make sure variables from other cells exist) ---
    try:
        assert os.path.exists(receptor_pdbqt)
        assert isinstance(ctr, tuple) and len(ctr) == 3
        assert len(lig_files) > 0
    except NameError:
        print("ERROR: 'receptor_pdbqt', 'ctr', or 'lig_files' not defined.")
        print("Please re-run the previous setup cells.")
        # This will stop the script if run in a fresh kernel
        # You can remove this 'try/except' if you are sure you re-ran them
    
    print(f"Starting parallel docking for {len(lig_files)} ligands using {N_WORKERS} workers...")
    start_time = time.time()
    
    # --- 3. Write CSV Header ---
    with open(out_csv, "w", newline="") as f:
        csv.writer(f).writerow(["tag", "smiles", "affinity", "pose"])

    # --- 4. Run Parallel Docking with Progress Bar ---
    # We use ProcessPoolExecutor for true CPU parallelism
    with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
        
        # Submit all jobs to the pool
        futures = [executor.submit(run_docking_task, job) for job in lig_files]
        
        # Use tqdm to create a progress bar as jobs are completed
        for future in tqdm(as_completed(futures), total=len(lig_files), desc="Docking Ligands"):
            
            result = future.result()
            
            # Write the result to the CSV *immediately* as it finishes
            with open(out_csv, "a", newline="") as f:
                csv.writer(f).writerow([
                    result["tag"], 
                    result["smiles"], 
                    result["affinity"], 
                    result["pose"]
                ])
                
            if result["status"] == "failed":
                print(f"[skip] {result['tag']}: {result['error']}")

    elapsed = (time.time() - start_time) / 60
    print(f"\nDocked {len(lig_files)} ligands in {elapsed:.1f} min → {out_csv}")

    # --- 5. Summarize Results ---
    try:
        dock_df = pd.read_csv(out_csv).dropna(subset=["affinity"]).sort_values("affinity")
        dock_df.to_csv("vina_scores_sorted.csv", index=False)
        print("\nAffinity summary (kcal/mol):")
        print(dock_df["affinity"].describe().round(3))
        print("\nTop-10 hits:")
        print(dock_df.head(10))
    except Exception as e:
        print(f"\nCould not summarize results: {e}")
        print("Please check the 'vina_scores_fixed.csv' file.")

#### Explanation for next cell

This cell performs post-docking analysis and comparison:
- Loads the sorted docking scores for de novo molecules (`vina_scores_sorted.csv`) and the ChEMBL HIV protease dataset (`chembl_hiv_protease_clean.csv`).
- Uses seaborn/matplotlib to plot the distribution of Vina affinities for the AI-generated molecules.
- Prints descriptive statistics for the de novo docking scores.
- Prints descriptive statistics for the `pchembl` values of the known inhibitors, providing a baseline to compare against the new candidates in the paper.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set a plotting style
sns.set(style="whitegrid")

# 1. Load your DE NOVO (AI-generated) valid scores
denovo_df = pd.read_csv("/kaggle/input/baseline-model/vina_scores_sorted.csv") # This is your 84.5-min run

# 2. Load the KNOWN DRUGS (your fine-tuning set)
baseline_df = pd.read_csv("/kaggle/input/baseline-model/chembl_hiv_protease_clean.csv")

# 3. Create the comparison plot
plt.figure(figsize=(10, 6))

# Plot a density plot (KDE) for your AI-generated scores
sns.kdeplot(denovo_df['affinity'], label="De Novo AI Molecules (Predicted Affinity)", 
            color='blue', fill=True)

# Vina scores for known drugs are often in a similar range.
# We can't plot the baseline's 'pchembl' on the same axis as 'affinity' 
# because the units are different.
# So, we just analyze the stats for our paper.

plt.title('Distribution of Predicted Binding Affinities for De Novo Molecules')
plt.xlabel('Vina Docking Score (kcal/mol)')
plt.ylabel('Density')
plt.legend()
plt.show()

# 4. Print the "Baseline" statistics for your paper
print("--- De Novo (AI) Results ---")
print(denovo_df['affinity'].describe().round(3))

print("\n--- Baseline (Known Drugs) Results ---")
print(baseline_df['pchembl'].describe().round(3))